# Static-topology 2×2, multi-seed (seeds 13, 17)

Closes the one remaining gap: the static-topology 2×2 was single-seed (seed 11), while every
other load-bearing result is 3-seed. This extends it to seeds 13 and 17 so the paper's headline
contrast — static topology ≈ 0 vs.\ dynamic flow gain — is uniformly multi-seed.

The 2×2 (stock \texttt{cudalstm}; only `use_basin_id_encoding` and the 5 topology static features
vary):

| | topology OFF | topology ON |
|---|---|---|
| one-hot ON | L | L+T |
| one-hot OFF | L_noID | L_noID+T |

Contrasts: `(L+T − L)` = does topology help the identity-encoded model (predict ≈0); `(L_noID+T −
L_noID)` = does it help when the model cannot memorize identity. Pre-registered prediction:
both ≈ 0 (static position is inert). L is already trained at all seeds, so this trains the three
new conditions per seed.

**Idempotent — Runtime → T4 GPU → Run all.** Skips any run already complete.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (SEEDS = [13, 17])

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEEDS=[13,17]  # seed 11 already done; these make the 2x2 3-seed
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found',c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'; os.makedirs(DRIVE_RUNS,exist_ok=True)
print('seeds', SEEDS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'),exist_ok=True)
print('symlinks ready')

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Generate the 5 topology static features

Writes `camels_attributes_v2.0/camels_topology.txt` (graph_depth, n_upstream, total_upstream_area,
in_degree, frac_upstream_area), which the L+T / L_noID+T configs load as static attributes.
Idempotent-safe to re-run (overwrites deterministically).

In [ ]:
%cd {REPO_DIR}
!python experiments/topology_ablation/generate_topology_attributes.py 2>&1 | tail -2

## Cell 8 — Run the 2×2 at seeds 13, 17

`run_2x2.py` builds the four configs and trains + evaluates each, skipping any already complete
(L is already trained at these seeds, so it trains L_T, L_noID, L_noID_T).

In [ ]:
%cd {REPO_DIR}
for SEED in SEEDS:
    print(f'\n########## 2x2 SEED {SEED} ##########')
    !python experiments/topology_ablation/run_2x2.py --networks component0 --seed {SEED} --device cuda:0 --epochs 30 2>&1 | tail -8

## Cell 9 — Verdict: 3-seed 2×2 contrasts (pooled)

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
B=f'{REPO_DIR}/runs/topology_ablation/component0'; ALL=[11]+SEEDS
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
def med(cond,s):
    x=nse(cond,s); return None if x is None else float(x.median())
def paired(a,b,s):
    xa,xb=nse(a,s),nse(b,s)
    if xa is None or xb is None: return None
    c=xa.index.intersection(xb.index); return float((xa.loc[c]-xb.loc[c]).median())
print('=== per-condition median NSE (per seed) ===')
for cond in ['L','L_T','L_noID','L_noID_T']:
    row=' / '.join(f'{med(cond,s):.3f}' if med(cond,s) is not None else '--' for s in ALL)
    print(f'  {cond:<10} {row}   (seeds {ALL})')
print('\n=== the two headline contrasts (median paired, per seed) ===')
for a,b,lbl in [('L_T','L','topo benefit WITH one-hot (predict ~0)'),
                ('L_noID_T','L_noID','topo benefit WITHOUT one-hot (predict ~0)')]:
    row=' / '.join(f'{paired(a,b,s):+.4f}' if paired(a,b,s) is not None else '--' for s in ALL)
    print(f'  ({a}-{b}) {lbl}\n     {row}   (seeds {ALL})')
# pooled significance of the WITHOUT-one-hot contrast (the one that could be >0)
d=[]
for s in ALL:
    xa,xb=nse('L_noID_T',s),nse('L_noID',s)
    if xa is None or xb is None: continue
    c=xa.index.intersection(xb.index); d+=list((xa.loc[c]-xb.loc[c]).values)
d=np.array(d)
if len(d):
    p2=wilcoxon(d,alternative='two-sided').pvalue
    print(f'\n(L_noID_T - L_noID) pooled: median {np.median(d):+.4f}, two-sided p={p2:.2f}, n={len(d)}')
    print('VERDICT: static topology is inert if both contrasts ~0 across seeds (|median| < ~0.01,'
          ' not significant). This is the paper''s static-null half, now multi-seed.')

## Cell 10 — Persistence check (did the new runs reach Drive?)

In [ ]:
%cd {REPO_DIR}
print('=== persistence (in Drive?) ===')
for s in SEEDS:
    for cond in ['L_T','L_noID','L_noID_T']:
        dp=f'{DRIVE_RUNS}/topology_ablation/component0/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
        print(f'  {cond}_seed{s}: {os.path.isfile(dp)}')

## Done

6 new runs persist to Drive (L_T / L_noID / L_noID_T × seeds 13, 17). Report the Cell 9 verdict +
Cell 10 persistence back; then the static-topology 2×2 is 3-seed and the paper's contrast is
uniformly multi-seed.